In [ ]:
# ==============================================================================
# CELL 1 / PHASE 1: DEWEY DECIMAL CLASSIFICATION (DDC) INGESTION
# 
# ARCHITECTURE NOTE: This script utilizes Strategy B (Live API Crawling) to 
# traverse the OCLC WorldCat Entities DDC service. It prioritize the native 
# alphanumeric hash as the primary key for CURIE resolvability while storing 
# the decimal notation as metadata. 
#
# QA RULE: Concepts labeled "[Unassigned]" are automatically excluded from 
# ingestion to maintain conceptual purity in the Bronze Layer.
# ==============================================================================

import os
import sys
import re
import requests
import pandas as pd
import time
from collections import deque
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
interim_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
error_log_dir = os.path.abspath(os.path.join(interim_data_dir, "logs", "errors"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(error_log_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)

try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py in config directory.")

# --- 2. Registry Lookup (DDC) ---
SOURCE_PREFIX = "DDC"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir,
    fallback_name="Dewey Decimal Classification",
    fallback_uri="https://id.oclc.org/worldcat/ddc/"
)

SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
hard_failure_file = os.path.join(error_log_dir, f"{SOURCE_PREFIX.lower()}_hard_failures.csv")

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 ({os.getenv("CONTACT_EMAIL", "Unspecified")})',
    'Accept': 'application/json'
}

# --- 3. Target Seeds & State Management ---
TARGET_SEEDS = [
    "https://id.oclc.org/worldcat/ddc/E3QGtWm7RBHVmJyQgtqjCv8TXt", # 200 Religion (Main Root)
    "https://id.oclc.org/worldcat/ddc/E3HM9M3F6KMqYF4pxBhDfQXjWJ", # 129 Origin and destiny of individual souls
    "https://id.oclc.org/worldcat/ddc/E3qj43xgv6XRqGKmBQbg3GctbX", # 130 Parapsychology & occultism
    "https://id.oclc.org/worldcat/ddc/E3hjRgRDJyKptrmrDvgxYd7pQC", # 140 Philosophical schools of thought
    "https://id.oclc.org/worldcat/ddc/E3MthvyKqbwqYRxrbcBfMq4XXk", # 366 Secret associations and societies
    "https://id.oclc.org/worldcat/ddc/E439MmWB9rxmcqBjf9wjK7TRry", # 390 Customs, etiquette & folklore
    "https://id.oclc.org/worldcat/ddc/E3R7mgDMjhhDcbfXmTptKYrjjY"  # 726 Buildings for religious and related purposes
]

if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    global_do_not_crawl = set(existing_df['URI'].dropna().tolist())
    print(f"[*] Resuming: {len(global_do_not_crawl)} concepts already in local Bronze Layer.")
else:
    global_do_not_crawl = set()

api_cache = {}
path_cache = {}
retry_queue = []

# --- 4. API & Utility Functions ---

def fetch_json(url):
    """Fetches JSON with 3-try retry loop and exponential backoff."""
    if url in api_cache: return api_cache[url]
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            if response.status_code == 200:
                data = response.json()
                api_cache[url] = data
                return data
            elif response.status_code in [429, 500, 502, 503, 504]:
                time.sleep(2 ** attempt)
        except requests.exceptions.RequestException:
            time.sleep(2 ** attempt)
    return None

def get_english_val(data_dict, field_name):
    """Extracts English strings/lists and normalizes whitespace."""
    if not data_dict or field_name not in data_dict: return ""
    field = data_dict[field_name]
    val = ""
    if isinstance(field, dict):
        en_content = field.get('en', "")
        val = " | ".join(en_content) if isinstance(en_content, list) else str(en_content)
    elif isinstance(field, list):
        val = " | ".join([str(v) for v in field])
    else:
        val = str(field)
    return re.sub(r'\s+', ' ', val).strip()

def get_best_path(uri):
    """BFS to construct Hierarchy_Path, filtering out [Unassigned] placeholders."""
    if uri in path_cache: return path_cache[uri]
    queue = deque([(uri, [])])
    visited = {uri}
    while queue:
        curr_uri, current_path = queue.popleft()
        data = fetch_json(curr_uri)
        if not data: continue
        label = get_english_val(data, 'prefLabel')
        # Skip [Unassigned] in breadcrumbs for semantic clarity
        new_path = ([label] if "[Unassigned]" not in label else []) + current_path
        broader_uri = data.get('broader')
        if not broader_uri:
            path_str = " > ".join(new_path)
            path_cache[uri] = path_str
            return path_str
        if broader_uri not in visited:
            visited.add(broader_uri)
            queue.append((broader_uri, new_path))
    return get_english_val(fetch_json(uri), 'prefLabel')

# --- 5. Extraction Logic ---

def process_node(uri, is_retry_pass=False):
    """Recursively parses downward. Excludes [Unassigned] concepts from CSV output."""
    data = fetch_json(uri)
    if not data:
        if not is_retry_pass: retry_queue.append(uri)
        return

    native_id = uri.rstrip('/').split('/')[-1]
    decimal_notation = str(data.get('notation', "")).strip()
    primary_label = get_english_val(data, 'prefLabel')

    # Data Filter: Exclude [Unassigned] placeholders from the Bronze Layer CSV
    if "[Unassigned]" not in primary_label and uri not in global_do_not_crawl:
        print(f"\r[{len(global_do_not_crawl)}] Ingesting: {decimal_notation} | {primary_label[:40]}...", end="", flush=True)

        parent_uri = data.get('broader', "")
        parent_id = parent_uri.rstrip('/').split('/')[-1] if parent_uri else ""
        crosswalks = data.get('closeMatch', [])
        if isinstance(crosswalks, str): crosswalks = [crosswalks]

        extracted_data = {
            'Source_System': SOURCE_NAME,
            'Primary_Label': primary_label,
            'CURIE': f"{SOURCE_PREFIX}:{native_id}",
            'Concept_Type': f"skos:{data.get('type', 'Concept')}",
            'Hierarchy_Path': get_best_path(uri),
            'Synonyms': decimal_notation,
            'Description': get_english_val(data, 'scopeNote'),
            'Parent_IDs': parent_id,
            'Concept_ID': native_id,
            'URI': uri,
            'Has_Translation': "yes" if len(data.get('prefLabel', {})) > 1 else "",
            'Status': 'active',
            'Crosswalks': " | ".join([str(c) for c in crosswalks])
        }

        # Finalize and append to CSV
        pd.DataFrame([finalize_row(extracted_data)]).to_csv(
            raw_ingest_file, mode='a', index=False, header=not os.path.exists(raw_ingest_file)
        )
        global_do_not_crawl.add(uri)
    else:
        # Node was either already ingested or is a placeholder to be ignored
        pass

    # Recurse Downward: Even if current node is [Unassigned], children may be valid
    narrower = data.get('narrower', [])
    narrower_list = [narrower] if isinstance(narrower, str) else narrower
    for child_uri in narrower_list:
        process_node(child_uri)

# --- 6. Execution ---
print(f"--- Starting Extraction: {SOURCE_NAME} ---")

for seed in TARGET_SEEDS:
    process_node(seed)

# Phase 2: Cleanup Pass
if retry_queue:
    print(f"\n[!] Phase 2 Cleanup: {len(retry_queue)} items...")
    active_retries = list(retry_queue)
    retry_queue.clear()
    for uri in active_retries:
        process_node(uri, is_retry_pass=True)

print(f"\n--- Extraction Complete. Data: {os.path.basename(raw_ingest_file)} ---")